#only use this on the jupyterhub first time use

#this is the training part

In [ ]:
from ultralytics import YOLO
import yaml
from datetime import datetime
import shutil
import os

timestamp = datetime.now().strftime("%Y_%m_%d_%H_%M_%S")

with open("settings.yaml", "r") as f:
    config = yaml.safe_load(f)

model = YOLO("yolo11n.pt") 

datasets = config['DATASET_LOCATIONS']
training_configurations = config['TRAINING_CONFIGURATIONS']

start_time = datetime.now()  # Start timing

for dataset in datasets:
    for configuration in training_configurations:
        # Only include keys that exist in configuration
        train_kwargs = {
            "data": dataset,
            "name": f"{timestamp}_model"
        }
        # Optional keys
        for key in ["time", "imgsz", "patience", "epochs", "multi_scale", "batch"]:
            if key in configuration:
                train_kwargs[key] = configuration[key]
        train_results = model.train(**train_kwargs)

        end_time = datetime.now()  # End timing
        duration = end_time - start_time
        total_minutes = int(duration.total_seconds() // 60)
        hours, minutes = divmod(total_minutes, 60)
        print(f"Training duration for: {hours}h {minutes}m")

#copy the model only after training, because of permissions
if os.path.isdir(f"runs/detect/{timestamp}_model"):
    source_directory = f"runs/detect/{timestamp}_model"
elif os.path.isdir(f"runs/segment/{timestamp}_model"):
    source_directory = f"runs/segment/{timestamp}_model"
else:
    print("No valid run directory found")

target_directory = f"{config['MODEL_PARENT_DIR']}/{config['DATASET_TYPE']}"
shutil.move(source_directory, target_directory)      



New https://pypi.org/project/ultralytics/8.3.191 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.3.190 🚀 Python-3.10.18 torch-2.8.0+cu128 CUDA:0 (NVIDIA RTX A4000, 15985MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/home/rolf/GIT/onemanstreasure/datasets/multiclass/der_merger-57422/dataset.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=Fa

PermissionError: [Errno 13] Permission denied: '/mnt/EBI-SHARE'

In [2]:
!nvidia-smi

Fri Aug  8 01:59:55 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.169                Driver Version: 570.169        CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 2070 ...    Off |   00000000:01:00.0 Off |                  N/A |
| N/A   56C    P8              4W /   80W |       6MiB /   8192MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [4]:
import cv2
import os
from pathlib import Path

def check_dataset_images(dataset_path):
    """Check for corrupted images in dataset"""
    corrupted_files = []
    
    for split in ['train', 'valid', 'test']:
        images_dir = Path(dataset_path) / 'images' / split
        if images_dir.exists():
            print(f"Checking {split} images...")
            for img_file in images_dir.glob('*'):
                if img_file.suffix.lower() in ['.jpg', '.jpeg', '.png', '.bmp']:
                    try:
                        img = cv2.imread(str(img_file))
                        if img is None:
                            corrupted_files.append(str(img_file))
                            print(f"Corrupted: {img_file}")
                    except Exception as e:
                        corrupted_files.append(str(img_file))
                        print(f"Error reading {img_file}: {e}")
    
    return corrupted_files

# Check your dataset
with open("settings.yaml", "r") as f:
    config = yaml.safe_load(f)

for dataset_path in config['DATASET_LOCATIONS']:
    print(f"\nChecking dataset: {dataset_path}")
    corrupted = check_dataset_images(dataset_path.replace('/dataset.yaml', ''))
    
    if corrupted:
        print(f"Found {len(corrupted)} corrupted files")
        # Remove corrupted files and their corresponding labels
        for img_path in corrupted:
            img_file = Path(img_path)
            # Remove image
            if img_file.exists():
                img_file.unlink()
                print(f"Removed: {img_file}")
            
            # Remove corresponding label file
            label_path = str(img_path).replace('/images/', '/labels/').replace(img_file.suffix, '.txt')
            label_file = Path(label_path)
            if label_file.exists():
                label_file.unlink()
                print(f"Removed label: {label_file}")
    else:
        print("No corrupted files found")


Checking dataset: /mnt/EBI-SHARE/06_Data/labelstudio/datasets/multiclass/ebi-rundgang-fused-qooqx-1-yolov8/data.yaml
No corrupted files found
